In [3]:
import pandas as pd
df = pd.read_csv("cleaned_reviews.csv")

In [ ]:
import requests
import json
import pandas as pd

# 1. Set your actual API Key here (inside the inverted commas "")
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# Updated the URL to use the recommended and latest 'gemini-3.6-flash' model
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.6-flash:generateContent?key={GOOGLE_API_KEY}"

def get_sentiment_and_aspect(review_text):
    # Handle blank text to avoid API errors
    if not isinstance(review_text, str) or len(review_text.strip()) == 0:
        return pd.Series({"Sentiment": "Neutral", "Category": "Unknown"})
        
    prompt = f"""
    Act as an AI analyzing a product review. Read the review below and extract two things:
    1. Sentiment: Is it Positive, Negative, or Neutral?
    2. Category: Dynamically extract the main aspect/category they are talking about (e.g., "Fragrance", "Battery", "Customer Service", "Skin Hydration"). Keep it short (1-2 words max).
    
    Review: "{review_text}"
    
    Return ONLY a valid JSON format like this:
    {{"Sentiment": "Positive", "Category": "Fragrance"}}
    """
    
    # Pack the prompt in the format Google's server expects
    payload = {
        "contents": [{"parts": [{"text": prompt}]}]
    }
    
    try:
        # Send a direct POST request to Gemini API
        response = requests.post(url, headers={'Content-Type': 'application/json'}, json=payload)
        response_data = response.json()
        
        # If Google sends back an error message, print it clearly
        if "error" in response_data:
            print(f"⚠️ API Error: {response_data['error']['message']}")
            return pd.Series({"Sentiment": "Error", "Category": "Error"})
            
        # Extract the AI's response text
        result_text = response_data['candidates'][0]['content']['parts'][0]['text']
        
        # Clean the result and convert it to JSON
        result_text = result_text.strip().replace("```json", "").replace("```", "")
        data = json.loads(result_text)
        
        return pd.Series({"Sentiment": data.get("Sentiment"), "Category": data.get("Category")})
    
    except Exception as e:
        # Print the exact error if it fails
        print(f"⚠️ Python Error on review: {e}")
        return pd.Series({"Sentiment": "Error", "Category": "Error"})

print("Connecting to the latest Gemini 3.6 Flash model... 🚀")

# We are testing on the first 5 rows only
test_df = df.head(5).copy()

# Apply the function to create two NEW columns at once
test_df[['AI_Sentiment', 'AI_Category']] = test_df['review_body'].apply(get_sentiment_and_aspect)

print("\nProcessing Complete!")
test_df[['review_body', 'AI_Sentiment', 'AI_Category']]

Connecting to the latest Gemini 3.6 Flash model... 🚀

Processing Complete!


,review_body,AI_Sentiment,AI_Category
0,This spray is really nice. It smells really go...,Positive,Hair Styling
1,"This product does what I need it to do, I just...",Negative,Fragrance
2,"Smells good, feels great!",Positive,Fragrance
3,Felt synthetic,Negative,Material
4,Love it,Positive,General Satisfaction


In [ ]:
import requests
import json
import pandas as pd
import time
from tqdm import tqdm

# ⚠️ APNI ASLI API KEY YAHAN ZAROOR DAALIYE

GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.6-flash:generateContent?key={GOOGLE_API_KEY}"

def get_sentiment_and_aspect(review_text):
    if not isinstance(review_text, str) or len(review_text.strip()) == 0:
        return {"Sentiment": "Neutral", "Category": "Unknown"}
        
    prompt = f"""
    Act as an AI analyzing a product review. Read the review below and extract two things:
    1. Sentiment: Is it Positive, Negative, or Neutral?
    2. Category: Dynamically extract the main aspect/category they are talking about (e.g., "Fragrance", "Battery", "Customer Service", "Skin Hydration"). Keep it short (1-2 words max).
    
    Review: "{review_text}"
    
    Return ONLY a valid JSON format like this:
    {{"Sentiment": "Positive", "Category": "Fragrance"}}
    """
    
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    
    for attempt in range(3):
        try:
            response = requests.post(url, headers={'Content-Type': 'application/json'}, json=payload)
            response_data = response.json()
            
            if "error" in response_data:
                error_msg = response_data['error']['message']
                # Ab hum error chupayenge nahi, seedha screen par print karenge!
                print(f"\n🚨 GOOGLE API ERROR: {error_msg}")
                
                # Agar rate limit hai toh wait karega
                if "429" in str(response_data) or "quota" in error_msg.lower() or "exhausted" in error_msg.lower():
                    print(f"⏳ Rate Limit... Waiting 30 seconds.")
                    time.sleep(30)
                    continue 
                return {"Sentiment": "Error", "Category": "Error"}
                
            result_text = response_data['candidates'][0]['content']['parts'][0]['text']
            result_text = result_text.strip().replace("```json", "").replace("```", "")
            data = json.loads(result_text)
            
            return {"Sentiment": data.get("Sentiment", "Neutral"), "Category": data.get("Category", "Unknown")}
        
        except Exception as e:
            # Agar Python level par koi error aaya toh wo yahan print hoga
            print(f"\n🚨 PYTHON ERROR on review '{review_text[:20]}...': {e}")
            if 'response_data' in locals():
                print(f"🔍 Raw Data causing issue: {response_data}")
            
    return {"Sentiment": "Error", "Category": "Error"} 

print("Running 8 Rows with FULL DEBUG LOGGING... 🚀")

sentiments = []
categories = []

for index, row in tqdm(df.head(8).iterrows(), total=8, desc="Analyzing"):
    result = get_sentiment_and_aspect(row['review_body'])
    sentiments.append(result['Sentiment'])
    categories.append(result['Category'])
    time.sleep(6)
    
test_df = df.head(8).copy()
test_df['AI_Sentiment'] = sentiments
test_df['AI_Category'] = categories

print("\nDone!")
test_df[['review_body', 'AI_Sentiment', 'AI_Category']]

Running 8 Rows with FULL DEBUG LOGGING... 🚀


Analyzing:  62%|██████▎   | 5/8 [01:32<00:44, 14.96s/it]


🚨 GOOGLE API ERROR: This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


Analyzing: 100%|██████████| 8/8 [02:09<00:00, 16.15s/it]


Done!


,review_body,AI_Sentiment,AI_Category
0,This spray is really nice. It smells really go...,Positive,Hair Texture
1,"This product does what I need it to do, I just...",Negative,Fragrance
2,"Smells good, feels great!",Positive,Fragrance
3,Felt synthetic,Negative,Material Quality
4,Love it,Positive,Overall
5,The polish was quiet thick and did not apply s...,Error,Error
6,Great for many tasks. I purchased these for m...,Positive,Makeup Removal
7,These were lightweight and soft but much too s...,Negative,Product Size
